### Настройка окружения и проверка подключения

In [46]:
import os
from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from enum import Enum
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

import time

import sqlite3

from langchain_core.runnables import RunnableLambda

from langchain_core.tools import tool

from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.exceptions import OutputParserException

In [9]:
os.environ["OPENAI_API_KEY"] = input("Введи API-токен: ")

In [10]:
model = ChatOpenAI(
    model="qwen3.8-27b",
    base_url="http://10.32.2.11:35000/v1",
    api_key=os.getenv("OPENAI_API_KEY"),
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)

In [11]:
response = model.invoke("Привет! Ответь одним коротким предложением.")
print(response.content)

Привет, я готов помочь вам.


### Часть 1. Базовая цепочка LangChain

Создаём шаблон промпта

In [12]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты полезный ассистент. Отвечай на языке {language}."),
    ("human", "{query}")
])

In [13]:
messages = prompt.format_messages(
    query="Объясни, что такое искусственный интеллект.",
    language="русском"
)

print(messages)

[SystemMessage(content='Ты полезный ассистент. Отвечай на языке русском.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Объясни, что такое искусственный интеллект.', additional_kwargs={}, response_metadata={})]


Собираем LCEL-цепочку

In [14]:
chain = prompt | model | StrOutputParser()

In [15]:
result = chain.invoke({
    "query": "Объясни, что такое искусственный интеллект.",
    "language": "русском"
})

print(result)

**Искусственный интеллект (ИИ)** — это область компьютерных наук, которая занимается созданием систем, способных выполнять задачи, обычно требующие человеческого интеллекта. Простыми словами, это программы, которые могут «думать», анализировать данные, принимать решения и даже общаться, имитируя когнитивные функции человека.

Вот ключевые аспекты, которые помогают понять суть ИИ:

### 1. Как это работает?
В основе большинства современных систем ИИ лежат **машинное обучение** и **глубокое обучение**.
*   **Машинное обучение**: Вместо того чтобы писать конкретные команды («если то, то это»), системе подают большие объемы данных (например, тысячи фотографий кошек). Алгоритм сам находит закономерности и учится распознавать кошек на новых фотографиях.
*   **Глубокое обучение**: Использует нейронные сети, вдохновленные устройством человеческого мозга, для решения сложных задач, таких как распознавание речи, перевод языков или генерация текста.

### 2. Виды искусственного интеллекта
*   **Узк

Сравниваем типы результатов

In [16]:
response = model.invoke("Что такое машинное обучение?")

print("Тип ответа модели:", type(response))
print("Тип ответа цепочки:", type(result))

Тип ответа модели: <class 'langchain_core.messages.ai.AIMessage'>
Тип ответа цепочки: <class 'langchain_core.messages.base.TextAccessor'>


Проверяем batch()

In [17]:
queries = [
    {"query": "Что такое NLP?", "language": "русском"},
    {"query": "Что такое большая языковая модель?", "language": "русском"},
    {"query": "Что такое токенизация текста?", "language": "русском"}
]

results = chain.batch(queries)

for result in results:
    print(result)
    print()

**NLP** — это аббревиатура, которая может означать две совершенно разные вещи в зависимости от контекста. Чаще всего в технологической среде под NLP понимают **Обработка естественного языка**, но в области психологии и личностного роста это **Нейролингвистическое программирование**.

Давайте разберем обе термины подробнее:

### 1. Обработка естественного языка (Natural Language Processing)
Это раздел искусственного интеллекта (AI) и лингвистики, который занимается взаимодействием между компьютерами и человеческим языком.

*   **Суть:** Компьютер учится понимать, интерпретировать и генерировать человеческий язык (текст или речь) так, как это делает человек.
*   **Примеры применения:**
    *   Чат-боты и виртуальные ассистенты (Siri, Alexa, Яндекс Алиса).
    *   Автоматический переводчик (Google Translate, DeepL).
    *   Анализ тональности (определение, позитивный или негативный отзыв о товаре).
    *   Генерация текстов (нейросети вроде меня, GPT, DALL-E).
    *   Поиск информации в и

**Выводы**
- Создан шаблон ChatPromptTemplate с системным и пользовательским сообщениями.
- С помощью LCEL собрана цепочка Prompt - LLM - StrOutputParser.
- Прямой вызов model.invoke() возвращает объект AIMessage, а цепочка chain.invoke() - строку str.
- Метод batch() успешно обработал три запроса и вернул три текстовых ответа.
- Разделение на компоненты позволяет удобно изменять промпт, модель и обработку результата независимо друг от друга.

### Часть 2. Structured Output и Pydantic

Опишем структуру данных

In [18]:
class ProductCategory(str, Enum):
    laptop = "laptop"
    smartphone = "smartphone"
    tablet = "tablet"
    other = "other"


class SortOrder(str, Enum):
    price_asc = "price_asc"
    price_desc = "price_desc"


class ProductQuery(BaseModel):
    category: ProductCategory = Field(
        default=ProductCategory.other,
        description="Категория товара"
    )
    max_price: int | None = Field(
        default=None,
        ge=0,
        description="Максимальная цена товара в рублях или null"
    )
    sort_by: SortOrder = Field(
        default=SortOrder.price_asc,
        description="Сортировка по цене: price_asc или price_desc"
    )

Создадим парсер

In [19]:
parser = PydanticOutputParser(pydantic_object=ProductQuery)
format_instructions = parser.get_format_instructions()
print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"ProductCategory": {"enum": ["laptop", "smartphone", "tablet", "other"], "title": "ProductCategory", "type": "string"}, "SortOrder": {"enum": ["price_asc", "price_desc"], "title": "SortOrder", "type": "string"}}, "properties": {"category": {"$ref": "#/$defs/ProductCategory", "default": "other", "description": "Категория товара"}, "max_price": {"anyOf": [{"minimum": 0, "type": "integer"}, {"type": "null"}], "default": null, "description": "Максимальная цена товара в рублях или null", "title": "Max Price"}, "sort_by": {"$ref": "#/$def

Создаём промпт и цепочку

In [20]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
Ты помощник для поиска товаров в каталоге.
Преобразуй запрос пользователя в параметры поиска.
Если категория не относится к ноутбукам, смартфонам или планшетам, используй other.
Если максимальная цена не указана, используй null.
Если пользователь просит самые дорогие товары, используй price_desc.
Если пользователь просит самые дешевые товары, используй price_asc.

{format_instructions}
"""),
    ("human", "{query}")
])

chain = prompt | model | parser

Тест на 5 обязательных запросах и еще 2 от себя  

In [21]:
queries = [
    "Найди ноутбуки дешевле 70000 рублей",
    "Покажи самые дорогие смартфоны",
    "Найди планшеты без ограничения по цене",
    "Подбери ноутбук стоимостью до 50 тысяч рублей",
    "Найди холодильники",
    "Хочу устройство для работы, чтобы было удобно брать его с собой",
    "Нужен гаджет для чтения книг и просмотра фильмов, желательно недорогой"
]

results = []

for query in queries:
    result = chain.invoke({
        "query": query,
        "format_instructions": format_instructions
    })

    results.append(result)

    print("Исходный запрос:", query)
    print("Результат:", result)
    print("JSON:", result.model_dump_json())
    print()

Исходный запрос: Найди ноутбуки дешевле 70000 рублей
Результат: category=<ProductCategory.laptop: 'laptop'> max_price=70000 sort_by=<SortOrder.price_asc: 'price_asc'>
JSON: {"category":"laptop","max_price":70000,"sort_by":"price_asc"}

Исходный запрос: Покажи самые дорогие смартфоны
Результат: category=<ProductCategory.smartphone: 'smartphone'> max_price=None sort_by=<SortOrder.price_desc: 'price_desc'>
JSON: {"category":"smartphone","max_price":null,"sort_by":"price_desc"}

Исходный запрос: Найди планшеты без ограничения по цене
Результат: category=<ProductCategory.tablet: 'tablet'> max_price=None sort_by=<SortOrder.price_asc: 'price_asc'>
JSON: {"category":"tablet","max_price":null,"sort_by":"price_asc"}

Исходный запрос: Подбери ноутбук стоимостью до 50 тысяч рублей
Результат: category=<ProductCategory.laptop: 'laptop'> max_price=50000 sort_by=<SortOrder.price_asc: 'price_asc'>
JSON: {"category":"laptop","max_price":50000,"sort_by":"price_asc"}

Исходный запрос: Найди холодильники
Р

Проверим поля объектов

In [22]:
for i in range(len(queries)):
    print("Запрос:", queries[i])
    print("Категория:", results[i].category.value)
    print("Максимальная цена:", results[i].max_price)
    print("Сортировка:", results[i].sort_by.value)
    print()

Запрос: Найди ноутбуки дешевле 70000 рублей
Категория: laptop
Максимальная цена: 70000
Сортировка: price_asc

Запрос: Покажи самые дорогие смартфоны
Категория: smartphone
Максимальная цена: None
Сортировка: price_desc

Запрос: Найди планшеты без ограничения по цене
Категория: tablet
Максимальная цена: None
Сортировка: price_asc

Запрос: Подбери ноутбук стоимостью до 50 тысяч рублей
Категория: laptop
Максимальная цена: 50000
Сортировка: price_asc

Запрос: Найди холодильники
Категория: other
Максимальная цена: None
Сортировка: price_asc

Запрос: Хочу устройство для работы, чтобы было удобно брать его с собой
Категория: laptop
Максимальная цена: None
Сортировка: price_asc

Запрос: Нужен гаджет для чтения книг и просмотра фильмов, желательно недорогой
Категория: tablet
Максимальная цена: None
Сортировка: price_asc



**Вывод:**
В ходе эксперимента модель успешно преобразовала семь текстовых запросов в структурированные объекты Pydantic. Во всех случаях значения соответствовали допустимым типам и перечислениям. При обработке неоднозначных запросов модель самостоятельно интерпретировала категорию товара и могла предполагать отсутствующие параметры, например максимальную цену. Это показывает, что валидация Pydantic проверяет структуру данных, но не гарантирует смысловую точность извлечённых параметров.

Сравнение моделей с размышлением и без

In [23]:
model_thinking = ChatOpenAI(
    model="qwen3.8-27b",
    base_url="http://10.32.2.11:35000/v1",
    api_key=os.getenv("OPENAI_API_KEY"),
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": True
        }
    }
)

In [24]:
chain_off = prompt | model | parser
chain_on = prompt | model_thinking | parser

In [25]:
test_queries = [
    "Хочу устройство для работы, чтобы было удобно брать его с собой",
    "Нужен гаджет для чтения книг и просмотра фильмов, желательно недорогой"
]

Выключенные размышления:

In [26]:
results_off = []

for query in test_queries:
    start_time = time.time()

    response = (prompt | model).invoke({
        "query": query,
        "format_instructions": format_instructions
    })

    end_time = time.time()

    results_off.append(response)

    print("Запрос:", query)
    print("Время:", round(end_time - start_time, 2), "сек.")
    print("Content:", response.content)
    print("Usage:", response.usage_metadata)
    print()

Запрос: Хочу устройство для работы, чтобы было удобно брать его с собой
Время: 0.31 сек.
Content: {
  "category": "laptop",
  "max_price": null,
  "sort_by": "price_asc"
}
Usage: {'input_tokens': 430, 'output_tokens': 31, 'total_tokens': 461, 'input_token_details': {}, 'output_token_details': {}}

Запрос: Нужен гаджет для чтения книг и просмотра фильмов, желательно недорогой
Время: 0.34 сек.
Content: ```json
{
  "category": "tablet",
  "max_price": null,
  "sort_by": "price_asc"
}
```
Usage: {'input_tokens': 432, 'output_tokens': 35, 'total_tokens': 467, 'input_token_details': {}, 'output_token_details': {}}



Включенные размышления:

In [27]:
results_on = []

for query in test_queries:
    start_time = time.time()

    response = (prompt | model_thinking).invoke({
        "query": query,
        "format_instructions": format_instructions
    })

    end_time = time.time()

    results_on.append(response)

    print("Запрос:", query)
    print("Время:", round(end_time - start_time, 2), "сек.")
    print("Content:", response.content)
    print("Additional kwargs:", response.additional_kwargs)
    print("Usage:", response.usage_metadata)
    print()

Запрос: Хочу устройство для работы, чтобы было удобно брать его с собой
Время: 8.67 сек.
Content: 

{
  "category": "other",
  "max_price": null,
  "sort_by": "price_asc"
}
Additional kwargs: {'refusal': None}
Usage: {'input_tokens': 466, 'output_tokens': 753, 'total_tokens': 1219, 'input_token_details': {}, 'output_token_details': {}}

Запрос: Нужен гаджет для чтения книг и просмотра фильмов, желательно недорогой
Время: 1.52 сек.
Content: 

```json
{
  "category": "tablet",
  "max_price": null,
  "sort_by": "price_asc"
}
```
Additional kwargs: {'refusal': None}
Usage: {'input_tokens': 468, 'output_tokens': 185, 'total_tokens': 653, 'input_token_details': {}, 'output_token_details': {}}



Теперь проверим, можно ли разобрать ответы обеих моделей через Pydantic-парсер:

In [28]:
print("Thinking OFF")

for response in results_off:
    try:
        result = parser.invoke(response.content)
        print(result)
    except Exception as error:
        print("Ошибка парсера:", error)

print()
print("Thinking ON")

for response in results_on:
    try:
        result = parser.invoke(response.content)
        print(result)
    except Exception as error:
        print("Ошибка парсера:", error)

Thinking OFF
category=<ProductCategory.laptop: 'laptop'> max_price=None sort_by=<SortOrder.price_asc: 'price_asc'>
category=<ProductCategory.tablet: 'tablet'> max_price=None sort_by=<SortOrder.price_asc: 'price_asc'>

Thinking ON
category=<ProductCategory.other: 'other'> max_price=None sort_by=<SortOrder.price_asc: 'price_asc'>
category=<ProductCategory.tablet: 'tablet'> max_price=None sort_by=<SortOrder.price_asc: 'price_asc'>


Проверка кол-ва выходных токенов:

In [29]:
for i in range(len(test_queries)):
    print("Запрос:", test_queries[i])

    print("OFF:", results_off[i].usage_metadata)
    print("ON:", results_on[i].usage_metadata)

    print()

Запрос: Хочу устройство для работы, чтобы было удобно брать его с собой
OFF: {'input_tokens': 430, 'output_tokens': 31, 'total_tokens': 461, 'input_token_details': {}, 'output_token_details': {}}
ON: {'input_tokens': 466, 'output_tokens': 753, 'total_tokens': 1219, 'input_token_details': {}, 'output_token_details': {}}

Запрос: Нужен гаджет для чтения книг и просмотра фильмов, желательно недорогой
OFF: {'input_tokens': 432, 'output_tokens': 35, 'total_tokens': 467, 'input_token_details': {}, 'output_token_details': {}}
ON: {'input_tokens': 468, 'output_tokens': 185, 'total_tokens': 653, 'input_token_details': {}, 'output_token_details': {}}



In [30]:
for i in range(len(results_on)):
    print("Запрос:", test_queries[i])
    print("Content:", results_on[i].content)
    print("Additional kwargs:", results_on[i].additional_kwargs)
    print()

Запрос: Хочу устройство для работы, чтобы было удобно брать его с собой
Content: 

{
  "category": "other",
  "max_price": null,
  "sort_by": "price_asc"
}
Additional kwargs: {'refusal': None}

Запрос: Нужен гаджет для чтения книг и просмотра фильмов, желательно недорогой
Content: 

```json
{
  "category": "tablet",
  "max_price": null,
  "sort_by": "price_asc"
}
```
Additional kwargs: {'refusal': None}



In [31]:
print("Thinking OFF")

for response in results_off:
    try:
        result = parser.invoke(response.content)
        print("Успешно:", result)
    except Exception as error:
        print("Ошибка:", error)

print("Thinking ON")

for response in results_on:
    try:
        result = parser.invoke(response.content)
        print("Успешно:", result)
    except Exception as error:
        print("Ошибка:", error)

Thinking OFF
Успешно: category=<ProductCategory.laptop: 'laptop'> max_price=None sort_by=<SortOrder.price_asc: 'price_asc'>
Успешно: category=<ProductCategory.tablet: 'tablet'> max_price=None sort_by=<SortOrder.price_asc: 'price_asc'>
Thinking ON
Успешно: category=<ProductCategory.other: 'other'> max_price=None sort_by=<SortOrder.price_asc: 'price_asc'>
Успешно: category=<ProductCategory.tablet: 'tablet'> max_price=None sort_by=<SortOrder.price_asc: 'price_asc'>


**Вывод:**
- Для первого запроса результаты различаются: размышления OFF определил ноутбук, размышления ON - категорию other.
- При включении thinking количество выходных токенов выросло примерно в 24,3 раза для первого запроса и в 5,3 раза для второго.
- Количество входных токенов также немного увеличилось.
- При выключенном thinking ответы содержат компактный JSON.
- В первом эксперименте модель определила ноутбук, во втором - планшет.
- В отличие от предыдущего эксперимента, здесь модель не стала самостоятельно назначать бюджет для второго запроса.

### Часть 3. Вызов классического программного обеспечения (SQLite)

Создадим db

In [3]:
conn = sqlite3.connect("products.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price INTEGER
)
""")

conn.commit()

Заполним db данными

In [4]:
products = [
    ("Lenovo IdeaPad 3", "laptop", 45000),
    ("ASUS VivoBook 15", "laptop", 55000),
    ("Acer Aspire 5", "laptop", 62000),
    ("HP Pavilion 15", "laptop", 68000),
    ("MacBook Air", "laptop", 120000),

    ("iPhone 15", "smartphone", 75000),
    ("Samsung Galaxy S24", "smartphone", 85000),
    ("Xiaomi Redmi Note 13", "smartphone", 18000),
    ("Google Pixel 8", "smartphone", 65000),
    ("Samsung Galaxy A55", "smartphone", 35000),

    ("iPad 10", "tablet", 40000),
    ("Samsung Galaxy Tab S9", "tablet", 70000),
    ("Xiaomi Pad 6", "tablet", 30000),
    ("Lenovo Tab M10", "tablet", 15000),
    ("Huawei MatePad", "tablet", 25000),

    ("LG 419 Refrigerator", "other", 34000)
]

cursor.executemany(
    "INSERT INTO products (name, category, price) VALUES (?, ?, ?)",
    products
)

conn.commit()

Проверим содержимое

In [5]:
cursor.execute("SELECT * FROM products")
data = cursor.fetchall()

for row in data:
    print(row)

(1, 'Lenovo IdeaPad 3', 'laptop', 45000)
(2, 'ASUS VivoBook 15', 'laptop', 55000)
(3, 'Acer Aspire 5', 'laptop', 62000)
(4, 'HP Pavilion 15', 'laptop', 68000)
(5, 'MacBook Air', 'laptop', 120000)
(6, 'iPhone 15', 'smartphone', 75000)
(7, 'Samsung Galaxy S24', 'smartphone', 85000)
(8, 'Xiaomi Redmi Note 13', 'smartphone', 18000)
(9, 'Google Pixel 8', 'smartphone', 65000)
(10, 'Samsung Galaxy A55', 'smartphone', 35000)
(11, 'iPad 10', 'tablet', 40000)
(12, 'Samsung Galaxy Tab S9', 'tablet', 70000)
(13, 'Xiaomi Pad 6', 'tablet', 30000)
(14, 'Lenovo Tab M10', 'tablet', 15000)
(15, 'Huawei MatePad', 'tablet', 25000)
(16, 'LG 419 Refrigerator', 'other', 34000)


Функция поиска товаров

In [6]:
def search_products(product_query: ProductQuery):
    if product_query.category == ProductCategory.other:
        return "Эта категория товаров не поддерживается."

    sql = "SELECT id, name, category, price FROM products WHERE category = ?"
    params = [product_query.category.value]

    if product_query.max_price is not None:
        sql += " AND price <= ?"
        params.append(product_query.max_price)

    if product_query.sort_by == SortOrder.price_asc:
        sql += " ORDER BY price ASC"
    else:
        sql += " ORDER BY price DESC"

    cursor.execute(sql, params)
    result = cursor.fetchall()

    return result

Проверка

In [32]:
test_query = ProductQuery(
    category="laptop",
    max_price=70000,
    sort_by="price_asc"
)

result = search_products(test_query)

for product in result:
    print(product)

(1, 'Lenovo IdeaPad 3', 'laptop', 45000)
(2, 'ASUS VivoBook 15', 'laptop', 55000)
(3, 'Acer Aspire 5', 'laptop', 62000)
(4, 'HP Pavilion 15', 'laptop', 68000)


Пороверка на 10 завпросах

In [33]:
test_queries = [
    ProductQuery(category="laptop"),
    ProductQuery(category="laptop", max_price=50000),
    ProductQuery(category="laptop", max_price=70000),
    ProductQuery(category="laptop", sort_by="price_desc"),
    ProductQuery(category="smartphone"),
    ProductQuery(category="smartphone", max_price=40000),
    ProductQuery(category="smartphone", sort_by="price_desc"),
    ProductQuery(category="tablet"),
    ProductQuery(category="tablet", max_price=20000),
    ProductQuery(category="tablet", max_price=10000),
    ProductQuery(category="other")
]

for query in test_queries:
    result = search_products(query)

    print("Параметры:", query.model_dump())
    print("Найдено:", result)
    print()

Параметры: {'category': <ProductCategory.laptop: 'laptop'>, 'max_price': None, 'sort_by': <SortOrder.price_asc: 'price_asc'>}
Найдено: [(1, 'Lenovo IdeaPad 3', 'laptop', 45000), (2, 'ASUS VivoBook 15', 'laptop', 55000), (3, 'Acer Aspire 5', 'laptop', 62000), (4, 'HP Pavilion 15', 'laptop', 68000), (5, 'MacBook Air', 'laptop', 120000)]

Параметры: {'category': <ProductCategory.laptop: 'laptop'>, 'max_price': 50000, 'sort_by': <SortOrder.price_asc: 'price_asc'>}
Найдено: [(1, 'Lenovo IdeaPad 3', 'laptop', 45000)]

Параметры: {'category': <ProductCategory.laptop: 'laptop'>, 'max_price': 70000, 'sort_by': <SortOrder.price_asc: 'price_asc'>}
Найдено: [(1, 'Lenovo IdeaPad 3', 'laptop', 45000), (2, 'ASUS VivoBook 15', 'laptop', 55000), (3, 'Acer Aspire 5', 'laptop', 62000), (4, 'HP Pavilion 15', 'laptop', 68000)]

Параметры: {'category': <ProductCategory.laptop: 'laptop'>, 'max_price': None, 'sort_by': <SortOrder.price_desc: 'price_desc'>}
Найдено: [(5, 'MacBook Air', 'laptop', 120000), (4, '

**Вывод:**
Создана SQLite-база данных с товарами трёх основных категорий. Реализована функция поиска, принимающая объект ProductQuery. Фильтрация выполняется по категории и максимальной цене, а сортировка по возрастанию или убыванию стоимости. 

Для защиты от некорректной подстановки значений используются параметризованные SQL-запросы. Проведено 11 тестов, включая проверку пустого результата и неподдерживаемой категории. Все тесты завершились ожидаемым образом.

Объединяем LLM и SQLite в одну цепочку

Подключаем RunnableLambda

In [35]:
product_chain = prompt | model | parser | RunnableLambda(search_products)

Проверяем цепочку

In [36]:
queries = [
    "Найди ноутбуки дешевле 70000 рублей",
    "Покажи самые дорогие смартфоны",
    "Найди планшеты дешевле 20000 рублей",
    "Найди холодильники"
]

for query in queries:
    result = product_chain.invoke({
        "query": query,
        "format_instructions": format_instructions
    })

    print("Запрос:", query)
    print("Результат:", result)
    print()

Запрос: Найди ноутбуки дешевле 70000 рублей
Результат: [(1, 'Lenovo IdeaPad 3', 'laptop', 45000), (2, 'ASUS VivoBook 15', 'laptop', 55000), (3, 'Acer Aspire 5', 'laptop', 62000), (4, 'HP Pavilion 15', 'laptop', 68000)]

Запрос: Покажи самые дорогие смартфоны
Результат: [(7, 'Samsung Galaxy S24', 'smartphone', 85000), (6, 'iPhone 15', 'smartphone', 75000), (9, 'Google Pixel 8', 'smartphone', 65000), (10, 'Samsung Galaxy A55', 'smartphone', 35000), (8, 'Xiaomi Redmi Note 13', 'smartphone', 18000)]

Запрос: Найди планшеты дешевле 20000 рублей
Результат: [(14, 'Lenovo Tab M10', 'tablet', 15000)]

Запрос: Найди холодильники
Результат: Эта категория товаров не поддерживается.



Сравниваем LLM без доступа к базе

In [37]:
for query in queries[:3]:
    result = chain.invoke({
        "query": query,
        "format_instructions": format_instructions
    })

    print("Запрос:", query)
    print("Параметры от LLM:", result)
    print()

Запрос: Найди ноутбуки дешевле 70000 рублей
Параметры от LLM: category=<ProductCategory.laptop: 'laptop'> max_price=70000 sort_by=<SortOrder.price_asc: 'price_asc'>

Запрос: Покажи самые дорогие смартфоны
Параметры от LLM: category=<ProductCategory.smartphone: 'smartphone'> max_price=None sort_by=<SortOrder.price_desc: 'price_desc'>

Запрос: Найди планшеты дешевле 20000 рублей
Параметры от LLM: category=<ProductCategory.tablet: 'tablet'> max_price=20000 sort_by=<SortOrder.price_asc: 'price_asc'>



Результаты показывают, что LCEL-цепочка корректно преобразует запрос в ProductQuery, а затем выполняет поиск в SQLite

Оформление поиска как инструмента

In [39]:
@tool("search_products", args_schema=ProductQuery)
def search_products_tool(
    category: ProductCategory,
    max_price: int | None = None,
    sort_by: SortOrder = SortOrder.price_asc
):
    """Ищет товары в каталоге SQLite по категории, максимальной цене и сортировке."""
    product_query = ProductQuery(
        category=category,
        max_price=max_price,
        sort_by=sort_by
    )
    return search_products(product_query)

In [40]:
model_with_tools = model.bind_tools([search_products_tool])

In [41]:
tool_response = model_with_tools.invoke(
    "Найди ноутбуки дешевле 70000 рублей"
)

print("Текст ответа:")
print(tool_response.content)

print("Вызовы инструментов:")
print(tool_response.tool_calls)

Текст ответа:

Вызовы инструментов:
[{'name': 'search_products', 'args': {'category': 'laptop', 'max_price': 70000, 'sort_by': 'price_asc'}, 'id': 'chatcmpl-tool-bceca76416e62099', 'type': 'tool_call'}]


Проверка аргументов и выполнение поиска

In [42]:
if tool_response.tool_calls:
    for tool_call in tool_response.tool_calls:
        if tool_call["name"] == "search_products":
            product_query = ProductQuery(**tool_call["args"])
            print("Проверенные параметры:")
            print(product_query)

            result = search_products(product_query)
            print("Результат поиска:")
            print(result)
else:
    print("Модель не вызвала инструмент.")

Проверенные параметры:
category=<ProductCategory.laptop: 'laptop'> max_price=70000 sort_by=<SortOrder.price_asc: 'price_asc'>
Результат поиска:
[(1, 'Lenovo IdeaPad 3', 'laptop', 45000), (2, 'ASUS VivoBook 15', 'laptop', 55000), (3, 'Acer Aspire 5', 'laptop', 62000), (4, 'HP Pavilion 15', 'laptop', 68000)]


Зафиксирован случай, когда модель передала "None" вместо null. Это показывает, почему аргументы инструмента необходимо валидировать до обращения к базе данных.

In [44]:
for query in test_queries:
    print("Запрос:", query)

    response = model_with_tools.invoke(query)

    if response.tool_calls:
        for tool_call in response.tool_calls:
            if tool_call["name"] == "search_products":
                try:
                    product_query = ProductQuery(**tool_call["args"])
                    result = search_products(product_query)

                    print("Параметры:", product_query)
                    print("Результат:", result)

                except Exception as error:
                    print("Ошибка проверки аргументов:", error)
    else:
        print("Модель не вызвала инструмент.")

    print()

Запрос: Найди ноутбуки дешевле 70000 рублей
Параметры: category=<ProductCategory.laptop: 'laptop'> max_price=70000 sort_by=<SortOrder.price_asc: 'price_asc'>
Результат: [(1, 'Lenovo IdeaPad 3', 'laptop', 45000), (2, 'ASUS VivoBook 15', 'laptop', 55000), (3, 'Acer Aspire 5', 'laptop', 62000), (4, 'HP Pavilion 15', 'laptop', 68000)]

Запрос: Покажи самые дорогие смартфоны
Параметры: category=<ProductCategory.smartphone: 'smartphone'> max_price=None sort_by=<SortOrder.price_desc: 'price_desc'>
Результат: [(7, 'Samsung Galaxy S24', 'smartphone', 85000), (6, 'iPhone 15', 'smartphone', 75000), (9, 'Google Pixel 8', 'smartphone', 65000), (10, 'Samsung Galaxy A55', 'smartphone', 35000), (8, 'Xiaomi Redmi Note 13', 'smartphone', 18000)]

Запрос: Найди планшеты без ограничения по цене
Параметры: category=<ProductCategory.tablet: 'tablet'> max_price=None sort_by=<SortOrder.price_asc: 'price_asc'>
Результат: [(14, 'Lenovo Tab M10', 'tablet', 15000), (15, 'Huawei MatePad', 'tablet', 25000), (13, 'X

**Небольщой вывод:** Вызов инструмента позволяет модели обращаться к базе данных, но сам по себе не гарантирует правильную интерпретацию неоднозначного запроса.

### Часть 4. Retry и журнал выполнения

Проверка ошибок парсера

In [45]:
invalid_outputs = [
    "Это не JSON",
    '{"category": "camera", "max_price": 10000, "sort_by": "price_asc"}',
    '{"category": "laptop", "max_price": -1000, "sort_by": "price_asc"}'
]

for output in invalid_outputs:
    print("Ответ модели:", output)

    try:
        result = parser.invoke(output)
        print("Результат:", result)
    except Exception as error:
        print("Ошибка:", type(error).__name__)
        print(error)

    print()

Ответ модели: Это не JSON
Ошибка: OutputParserException
Invalid json output: Это не JSON
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

Ответ модели: {"category": "camera", "max_price": 10000, "sort_by": "price_asc"}
Ошибка: OutputParserException
Failed to parse ProductQuery from completion {"category": "camera", "max_price": 10000, "sort_by": "price_asc"}. Got: 1 validation error for ProductQuery
category
  Input should be 'laptop', 'smartphone', 'tablet' or 'other' [type=enum, input_value='camera', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/enum
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

Ответ модели: {"category": "laptop", "max_price": -1000, "sort_by": "price_asc"}
Ошибка: OutputParserException
Failed to parse ProductQuery from completion {"category": "laptop", "max_price": -1000, "sort_by": "price_asc"}. Got: 1 vali

Pydantic обнаруживает нарушения, а PydanticOutputParser преобразует их в OutputParserException. Это и будем использовать для повторных попыток

In [47]:
fake_model = FakeListChatModel(
    responses=[
        "Это не JSON",
        '{"category": "laptop", "max_price": 50000, "sort_by": "price_asc"}'
    ]
)

fake_chain = prompt | fake_model | parser

retry_chain = fake_chain.with_retry(
    stop_after_attempt=3,
    retry_if_exception_type=(OutputParserException,)
)

result = retry_chain.invoke({
    "query": "Найди ноутбуки дешевле 50000 рублей",
    "format_instructions": format_instructions
})

print(result)

category=<ProductCategory.laptop: 'laptop'> max_price=50000 sort_by=<SortOrder.price_asc: 'price_asc'>


Теперь проверим случай, когда все три попытки неудачные

In [48]:
fake_model_bad = FakeListChatModel(
    responses=[
        "Это не JSON",
        '{"category": "camera", "max_price": 10000, "sort_by": "price_asc"}',
        '{"category": "laptop", "max_price": -1000, "sort_by": "price_asc"}'
    ]
)

fake_chain_bad = prompt | fake_model_bad | parser

retry_chain_bad = fake_chain_bad.with_retry(
    stop_after_attempt=3,
    retry_if_exception_type=(OutputParserException,)
)

try:
    result = retry_chain_bad.invoke({
        "query": "Найди ноутбуки дешевле 50000 рублей",
        "format_instructions": format_instructions
    })
    print(result)
except OutputParserException as error:
    print("Все три попытки завершились ошибкой.")
    print(type(error).__name__)
    print(error)

Все три попытки завершились ошибкой.
OutputParserException
Failed to parse ProductQuery from completion {"category": "laptop", "max_price": -1000, "sort_by": "price_asc"}. Got: 1 validation error for ProductQuery
max_price
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-1000, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/greater_than_equal
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 


Сравниваем с повторными попытками и без них

In [49]:
fake_model_no_retry = FakeListChatModel(
    responses=[
        "Это не JSON",
        '{"category": "laptop", "max_price": 50000, "sort_by": "price_asc"}'
    ]
)

chain_no_retry = prompt | fake_model_no_retry | parser

try:
    result = chain_no_retry.invoke({
        "query": "Найди ноутбуки дешевле 50000 рублей",
        "format_instructions": format_instructions
    })
    print("Без retry:", result)
except OutputParserException:
    print("Без retry: ошибка парсинга.")

fake_model_retry = FakeListChatModel(
    responses=[
        "Это не JSON",
        '{"category": "laptop", "max_price": 50000, "sort_by": "price_asc"}'
    ]
)

chain_retry = (prompt | fake_model_retry | parser).with_retry(
    stop_after_attempt=3,
    retry_if_exception_type=(OutputParserException,)
)

result = chain_retry.invoke({
    "query": "Найди ноутбуки дешевле 50000 рублей",
    "format_instructions": format_instructions
})

print("С retry:", result)

Без retry: ошибка парсинга.
С retry: category=<ProductCategory.laptop: 'laptop'> max_price=50000 sort_by=<SortOrder.price_asc: 'price_asc'>


**Итого получили:**
- без retry - OutputParserException;
- с retry - ProductQuery(category=laptop, max_price=50000, sort_by=price_asc);
- после трёх неудачных попыток - ошибка возвращается вызывающему коду.

### Часть 5. Уточняющие запросы

Создаём цепочку с передачей предыдущих параметров

In [50]:
prompt_state = ChatPromptTemplate.from_messages([
    ("system", """
Ты помощник для поиска товаров.
Пользователь уточняет предыдущий запрос.

Текущие параметры поиска:
{current_query}

Обнови параметры с учетом нового сообщения.
Если пользователь не меняет какое-либо поле, сохрани его прежнее значение.
Не придумывай ограничения и значения, которых пользователь не указал.

{format_instructions}
"""),
    ("human", "{query}")
])

state_chain = prompt_state | model | parser

Првоерим диалог из трёх сообщений

In [54]:
current_query = ProductQuery(
    category=ProductCategory.laptop,
    max_price=70000,
    sort_by=SortOrder.price_asc
)

query = "Теперь только до 50000 рублей"

result = state_chain.invoke({
    "current_query": current_query.model_dump_json(),
    "query": query,
    "format_instructions": format_instructions
})

current_query = result
print(current_query)

query = "Покажи от самых дорогих"

result = state_chain.invoke({
    "current_query": current_query.model_dump_json(),
    "query": query,
    "format_instructions": format_instructions
})

current_query = result
print(current_query)

category=<ProductCategory.laptop: 'laptop'> max_price=50000 sort_by=<SortOrder.price_asc: 'price_asc'>
category=<ProductCategory.laptop: 'laptop'> max_price=50000 sort_by=<SortOrder.price_desc: 'price_desc'>


In [56]:
current_query = ProductQuery(
    category=ProductCategory.tablet,
    max_price=None,
    sort_by=SortOrder.price_asc
)

query = "Покажи сначала самые дорогие"

result = state_chain.invoke({
    "current_query": current_query.model_dump_json(),
    "query": query,
    "format_instructions": format_instructions
})

current_query = result
print("После первого уточнения:", current_query)

query = "А теперь смартфоны"

result = state_chain.invoke({
    "current_query": current_query.model_dump_json(),
    "query": query,
    "format_instructions": format_instructions
})

current_query = result
print("После второго уточнения:", current_query)

После первого уточнения: category=<ProductCategory.tablet: 'tablet'> max_price=None sort_by=<SortOrder.price_desc: 'price_desc'>
После второго уточнения: category=<ProductCategory.smartphone: 'smartphone'> max_price=None sort_by=<SortOrder.price_desc: 'price_desc'>


**Итого:** оба диалога с хранением состояния прошли успешно: параметры, которые мы не меняли, сохранялись

Сравним полную историю и хранение состояния

In [57]:
history_prompt = ChatPromptTemplate.from_messages([
    ("system", """
Ты помощник для поиска товаров.
По истории диалога определи актуальные параметры поиска.
Сохраняй прежние значения, если пользователь их не меняет.

{format_instructions}
"""),
    ("human", "{history}")
])

In [58]:
current_query = ProductQuery(
    category=ProductCategory.laptop,
    max_price=50000,
    sort_by=SortOrder.price_asc
)

In [60]:
history = """
Пользователь: Найди ноутбуки дешевле 70000 рублей
Ассистент: {"category": "laptop", "max_price": 70000, "sort_by": "price_asc"}

Пользователь: Теперь только до 50000 рублей
Ассистент: {"category": "laptop", "max_price": 50000, "sort_by": "price_asc"}

Пользователь: Покажи от самых дорогих
"""

history_messages = history_prompt.invoke({
    "history": history,
    "format_instructions": format_instructions
})

history_response = model.invoke(history_messages)
history_result = parser.invoke(history_response.content)

print("Полная история:")
print(history_result)
print("Входные токены:", history_response.usage_metadata["input_tokens"])

state_messages = prompt_state.invoke({
    "current_query": current_query.model_dump_json(),
    "query": "Покажи от самых дорогих",
    "format_instructions": format_instructions
})

state_response = model.invoke(state_messages)
state_result = parser.invoke(state_response.content)

print("Хранение состояния:")
print(state_result)
print("Входные токены:", state_response.usage_metadata["input_tokens"])

Полная история:
category=<ProductCategory.laptop: 'laptop'> max_price=50000 sort_by=<SortOrder.price_desc: 'price_desc'>
Входные токены: 479
Хранение состояния:
category=<ProductCategory.laptop: 'laptop'> max_price=50000 sort_by=<SortOrder.price_desc: 'price_desc'>
Входные токены: 434


Сравниваем второй диалог

In [61]:
history = """
Пользователь: Найди планшеты без ограничения по цене
Ассистент: {"category": "tablet", "max_price": null, "sort_by": "price_asc"}

Пользователь: Покажи сначала самые дорогие
Ассистент: {"category": "tablet", "max_price": null, "sort_by": "price_desc"}

Пользователь: А теперь смартфоны
"""

history_messages = history_prompt.invoke({
    "history": history,
    "format_instructions": format_instructions
})

history_response = model.invoke(history_messages)
history_result = parser.invoke(history_response.content)

print("Полная история:")
print(history_result)
print("Входные токены:", history_response.usage_metadata["input_tokens"])

current_query = ProductQuery(
    category=ProductCategory.tablet,
    max_price=None,
    sort_by=SortOrder.price_desc
)

state_messages = prompt_state.invoke({
    "current_query": current_query.model_dump_json(),
    "query": "А теперь смартфоны",
    "format_instructions": format_instructions
})

state_response = model.invoke(state_messages)
state_result = parser.invoke(state_response.content)

print("Хранение состояния:")
print(state_result)
print("Входные токены:", state_response.usage_metadata["input_tokens"])

Полная история:
category=<ProductCategory.smartphone: 'smartphone'> max_price=None sort_by=<SortOrder.price_desc: 'price_desc'>
Входные токены: 457
Хранение состояния:
category=<ProductCategory.smartphone: 'smartphone'> max_price=None sort_by=<SortOrder.price_desc: 'price_desc'>
Входные токены: 426


**Вывод:** хранение текущего состояния позволило сохранить корректность параметров и уменьшить расход входных токенов в обоих тестовых диалогах. В этих экспериментах экономия составила порядка 8%.